# CS598JBR MP4 - agentic C-to-Rust translation (Team 12)
Model: a free hosted open-weight model on NVIDIA's API catalog (`configs/config.nvidia.json`). The catalog changes over time, so section 2 lists the models available to the key and picks the first preferred one that supports tool calling (Qwen3-Coder, Kimi-K2, DeepSeek-V3, GLM, ...). A CPU runtime is enough.
Before running: create a free key at build.nvidia.com and add it as a Colab secret named `NVIDIA_API_KEY` (key icon in the left sidebar, enable notebook access).
If the runtime disconnects: re-run sections 0-3, skip section 4, then run the `--resume` cell.

## 0. Clone the repo

In [ ]:
%cd /content
!git clone https://github.com/picolimpid/CS598JBR-Team-12.git
%cd CS598JBR-Team-12/MP4
!git pull
seed = "191374802308824446074400153398805442526"   # MP3 leftover, not used by MP4
humaneval_input_dataset = "selected_humaneval_" + seed + ".jsonl"

## 1. Rust toolchain nightly-2025-05-13 (+clippy, rustfmt) and the newmetrics unsafe-metrics tool (~5-10 min)

In [ ]:
import os
!bash scripts/setup_colab.sh
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
!cargo +nightly-2025-05-13 --version && cargo +nightly-2025-05-13 clippy --version && which cargo-newmetrics

## 2. Load the API key from Colab secrets and pick a model (the key is never printed or written to disk)

In [ ]:
from google.colab import userdata

os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
CONFIG = "configs/config.nvidia.json"

# list the models this key can use, try them in the order of model_preferences, keep the first that calls tools
out = !python3 src/pipeline.py --project 1 --config {CONFIG} --check-llm
print("\n".join(out))
MODEL = next((l.split("selected model: ", 1)[1].strip() for l in out if l.startswith("selected model: ")), None)
print("MODEL =", MODEL)   # both projects use this model

## 3. Back up results to Google Drive every 5 minutes
After a disconnect this cell first restores the saved progress from Drive, so `--resume` can continue.

In [ ]:
import subprocess
from google.colab import drive
drive.mount("/content/drive")
BACKUP = "/content/drive/MyDrive/CS598JBR_MP4_backup"
os.makedirs(BACKUP, exist_ok=True)
os.makedirs("translations", exist_ok=True)
os.makedirs("trajectories", exist_ok=True)
!command -v rsync > /dev/null || apt-get -qq install -y rsync > /dev/null

# restore progress saved before a disconnect
for p in ("project_1", "project_2"):
    if not os.path.exists(f"trajectories/{p}/state.json") and os.path.exists(f"{BACKUP}/trajectories/{p}/state.json"):
        print("restoring", p, "from Drive")
        !rsync -a "{BACKUP}/trajectories/{p}" trajectories/
        !rsync -a --exclude target/ "{BACKUP}/translations/{p}" translations/

loop = f"while true; do rsync -a --exclude target/ translations trajectories '{BACKUP}/'; sleep 300; done"
if subprocess.run(["pgrep", "-f", "CS598JBR_MP4_backup"], capture_output=True).returncode != 0:
    subprocess.Popen(["bash", "-c", loop], start_new_session=True)
print("backing up translations/ and trajectories/ to", BACKUP, "every 5 min")

## 4. Smoke test: build the crate skeleton + baseline cargo/clippy/newmetrics checks (no LLM)
Skip this cell after a disconnect: it resets `translations/`.

In [ ]:
!python3 src/pipeline.py --project 1 --config {CONFIG} --dry-run
!python3 src/pipeline.py --project 2 --config {CONFIG} --dry-run

## 5. Run the agent on project_1 (satc). Keep the tab open.

In [ ]:
!python3 src/pipeline.py --project 1 --config {CONFIG} --model {MODEL}

## 6. Run the agent on project_2 (jccc)

In [ ]:
!python3 src/pipeline.py --project 2 --config {CONFIG} --model {MODEL}

Only after a disconnect: re-run sections 0-3 (section 3 restores the progress from Drive), skip section 4, then continue the interrupted project with `--resume` (change 1 -> 2 as needed).

In [ ]:
# !python3 src/pipeline.py --project 1 --config {CONFIG} --model {MODEL} --resume

## 7. Final evaluation: build, tests, Clippy per group, newmetrics -> tables for the report

In [ ]:
!python3 src/evaluate.py --config {CONFIG}
!cat trajectories/project_1/summary.md trajectories/project_2/summary.md

## 8. Download the results (translations + trajectories), then commit them to the repo locally

In [ ]:
!zip -qr /content/mp4_results.zip translations trajectories -x '*/target/*'
from google.colab import files
files.download("/content/mp4_results.zip")